# Aerial Airport Point RL Pipeline

Run order:
1. Install dependencies if needed.
2. Build the dataset from `aerial_airport/raw_dataset/Aerial Airport.coco`.
3. Launch the 4 round-2 recall configs in parallel.
4. Check status or wait for all runs to finish.
5. Optionally push the built dataset to HF.
6. Benchmark the 4 selected checkpoints on the built `test` split.


In [1]:
import os
from pathlib import Path

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / "aerial_airport").exists() and (candidate / "MDpi_and_d").exists():
        os.chdir(candidate)
        break

print(Path.cwd())


/Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo


In [2]:
# Install once if needed.
!python -m pip install -q datasets pillow numpy scipy wandb python-dotenv


## 1) Build the Local Dataset from the Raw COCO Export

This writes the normalized dataset to `aerial_airport/outputs/maxs-m87_aerial_airport_point_v2`.
The current build config auto-splits the single raw `train/` export into `train`, `validation`, and `test`, then tops up each split toward a `10%` empty-row target.


In [3]:
!python aerial_airport/build_aerial_airport_hf_dataset.py \
  --config aerial_airport/configs/build_aerial_airport_hf_dataset_default.json \
  --push-to-hub ""


Saving the dataset (1/1 shards): 100%|█| 296/296 [00:00<00:00, 3916.66 examples/
Saving the dataset (1/1 shards): 100%|█| 37/37 [00:00<00:00, 3414.28 examples/s]
saved normalized dataset to /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/aerial_airport/outputs/maxs-m87_aerial_airport_point_v2 (train=296, validation=38, test=37)


## 2) Run All Round-2 Recall Experiments in Parallel

This launches the 4 round-2 recall configs in parallel. Each run writes to its own log file under `aerial_airport/logs/round2_parallel/`.

The configs rotate across staging API keys:
- `lr1e4-r8`: `CICID_GPUB_MOONDREAM_API_KEY_1`
- `lr5e4-r8`: `CICID_GPUB_MOONDREAM_API_KEY_2`
- `lr1e4-r16`: `CICID_GPUB_MOONDREAM_API_KEY_3`
- `lr5e4-r16`: `CICID_GPUB_MOONDREAM_API_KEY_4`


In [4]:
import subprocess
import sys
from pathlib import Path

parallel_configs = [
    "aerial_airport/configs/cicd/cicd_train_aerial_airport_point_recall_lr1e4_r8.json",
    "aerial_airport/configs/cicd/cicd_train_aerial_airport_point_recall_lr5e4_r8.json",
    "aerial_airport/configs/cicd/cicd_train_aerial_airport_point_recall_lr1e4_r16.json",
    "aerial_airport/configs/cicd/cicd_train_aerial_airport_point_recall_lr5e4_r16.json",
]

logs_dir = Path("aerial_airport/logs/round2_parallel")
logs_dir.mkdir(parents=True, exist_ok=True)

parallel_train_runs = []
for config_path in parallel_configs:
    config_name = Path(config_path).stem
    log_path = logs_dir / f"{config_name}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    cmd = [
        sys.executable,
        "aerial_airport/train_aerial_airport_point.py",
        "--config",
        config_path,
    ]
    proc = subprocess.Popen(cmd, stdout=log_handle, stderr=subprocess.STDOUT)
    parallel_train_runs.append(
        {
            "name": config_name,
            "config_path": config_path,
            "pid": proc.pid,
            "process": proc,
            "log_path": str(log_path),
            "log_handle": log_handle,
        }
    )
    print(f"started {config_name} | pid={proc.pid} | log={log_path}")

print(f"launched {len(parallel_train_runs)} parallel training runs")


started cicd_train_aerial_airport_point_recall_lr1e4_r8 | pid=19298 | log=aerial_airport/logs/round2_parallel/cicd_train_aerial_airport_point_recall_lr1e4_r8.log
started cicd_train_aerial_airport_point_recall_lr5e4_r8 | pid=19299 | log=aerial_airport/logs/round2_parallel/cicd_train_aerial_airport_point_recall_lr5e4_r8.log
started cicd_train_aerial_airport_point_recall_lr1e4_r16 | pid=19300 | log=aerial_airport/logs/round2_parallel/cicd_train_aerial_airport_point_recall_lr1e4_r16.log
started cicd_train_aerial_airport_point_recall_lr5e4_r16 | pid=19301 | log=aerial_airport/logs/round2_parallel/cicd_train_aerial_airport_point_recall_lr5e4_r16.log
launched 4 parallel training runs


## 3) Check Parallel Run Status and Tail Recent Logs

Run this while the jobs are active to see return codes and the last few log lines for each process.


In [5]:
if "parallel_train_runs" not in globals() or not parallel_train_runs:
    raise RuntimeError("Launch the parallel runs first.")

for run in parallel_train_runs:
    proc = run["process"]
    return_code = proc.poll()
    status = "running" if return_code is None else f"finished rc={return_code}"
    print(f"\n[{run['name']}] pid={run['pid']} status={status}")
    log_path = Path(run["log_path"])
    if log_path.exists():
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        for line in lines[-10:]:
            print(line)



[cicd_train_aerial_airport_point_recall_lr1e4_r8] pid=19298 status=running

[cicd_train_aerial_airport_point_recall_lr5e4_r8] pid=19299 status=running

[cicd_train_aerial_airport_point_recall_lr1e4_r16] pid=19300 status=running

[cicd_train_aerial_airport_point_recall_lr5e4_r16] pid=19301 status=running


## 4) Wait For All Parallel Runs To Finish

This blocks until every launched process exits, then closes the log handles.


In [6]:
if "parallel_train_runs" not in globals() or not parallel_train_runs:
    raise RuntimeError("Launch the parallel runs first.")

for run in parallel_train_runs:
    rc = run["process"].wait()
    run["log_handle"].close()
    print(f"finished {run['name']} with rc={rc}")


finished cicd_train_aerial_airport_point_recall_lr1e4_r8 with rc=0
finished cicd_train_aerial_airport_point_recall_lr5e4_r8 with rc=0
finished cicd_train_aerial_airport_point_recall_lr1e4_r16 with rc=0
finished cicd_train_aerial_airport_point_recall_lr5e4_r16 with rc=0


## 5) Optional HF Push

Run this if you want to publish the normalized dataset and then train by `dataset_name` instead of `dataset_path`.


In [7]:
!python aerial_airport/build_aerial_airport_hf_dataset.py \
  --config aerial_airport/configs/build_aerial_airport_hf_dataset_default.json \
  --push-to-hub maxs-m87/aerial_airport_point_v2


Saving the dataset (1/1 shards): 100%|█| 296/296 [00:00<00:00, 2502.03 examples/
Saving the dataset (1/1 shards): 100%|█| 37/37 [00:00<00:00, 2187.43 examples/s]
saved normalized dataset to /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/aerial_airport/outputs/maxs-m87_aerial_airport_point_v2 (train=296, validation=38, test=37)
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Map: 100%|███████████████████████████| 296/296 [00:00<00:00, 9203.15 examples/s]

Creating parquet from Arrow format: 100%|█████████| 1/1 [00:00<00:00, 27.37ba/s]
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...0gn/T/tmplzpnq56p.parquet:   3%|▍             |  525kB / 17.3MB            

Processing Files (0 / 1)      :   3%|▍             |  525kB / 17.3MB,  375kB/s  
New Data Upload               :   3%|▍             |  525kB 

## 6) Benchmark the 4 Selected Checkpoints on the Built `test` Split

Fill in the best checkpoint from each of the 4 training runs, then run the sequential benchmark cell. Each benchmark writes a separate JSON file under `aerial_airport/outputs/benchmarks/`.


In [8]:
benchmark_targets = [
    {"label": "lr1e4-r8", "finetune_id": "replace-me", "checkpoint_step": 200},
    {"label": "lr5e4-r8", "finetune_id": "replace-me", "checkpoint_step": 200},
    {"label": "lr1e4-r16", "finetune_id": "replace-me", "checkpoint_step": 200},
    {"label": "lr5e4-r16", "finetune_id": "replace-me", "checkpoint_step": 200},
]


In [9]:
import subprocess
import sys
from pathlib import Path

bench_dir = Path("aerial_airport/outputs/benchmarks")
bench_dir.mkdir(parents=True, exist_ok=True)

for item in benchmark_targets:
    if item["finetune_id"] == "replace-me":
        raise RuntimeError(f"Set finetune_id for {item['label']} before benchmarking.")
    out_json = bench_dir / f"benchmark_{item['label']}.json"
    cmd = [
        sys.executable,
        "aerial_airport/benchmark_aerial_airport_point.py",
        "--config",
        "aerial_airport/configs/benchmark_aerial_airport_point_default.json",
        "--finetune-id",
        item["finetune_id"],
        "--checkpoint-step",
        str(item["checkpoint_step"]),
        "--out-json",
        str(out_json),
    ]
    print("running", item["label"], "->", out_json)
    subprocess.run(cmd, check=True)


RuntimeError: Set finetune_id for lr1e4-r8 before benchmarking.